In [1]:
import os, json, time, datetime as dt, csv, pathlib
from typing import Dict, List
import requests
import pandas as pd
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from pathlib import Path

In [2]:
from pathlib import Path

ROOT = Path.cwd()          # notebooks are meant to be run from their own folder
CHECKS = [
    (".env", "NEEDED", "YOU create this: copy .env.example to .env. Missing = no error, but the config demo silently shows nothing"),
    (".env.example", "NEEDED", "shipped with this stage - the template you copy to .env"),
]

print(f"Looking in: {ROOT}\n")
missing = 0
for rel, kind, note in CHECKS:
    here = (ROOT / rel).exists()
    if not here and kind == "NEEDED":
        missing += 1
    print(f"  [{'OK ' if here else 'MISS'}]  {kind:<8}  {rel:<34}  {note}")

if missing:
    print(f"\n{missing} needed file(s) missing. Put them at the paths above, relative to:\n  {ROOT}")
    print("If that folder looks wrong, you are running the notebook from the wrong place.")
else:
    print("\nAll needed files present.")

Looking in: c:\Users\schwa\bootcamp_michael_brick\homework\homework04

  [OK ]  NEEDED    .env                                YOU create this: copy .env.example to .env. Missing = no error, but the config demo silently shows nothing
  [OK ]  NEEDED    .env.example                        shipped with this stage - the template you copy to .env

All needed files present.


In [3]:
DATA_RAW = pathlib.Path("data/raw")
DATA_RAW.mkdir(parents=True, exist_ok=True)

load_dotenv()

ALPHA_KEY = os.getenv("ALPHAVANTAGE_API_KEY")
print("Loaded ALPHAVANTAGE_API_KEY?", bool(ALPHA_KEY))

Loaded ALPHAVANTAGE_API_KEY? True


In [4]:
def safe_stamp():
    return dt.datetime.now().strftime("%Y%m%d-%H%M%S")
def safe_filename(prefix: str, meta: Dict[str, str]) -> str:
    mid = "_".join(
        [f"{k}-{str(v).replace(' ', '-')[:20]}" for k, v in meta.items()]
    )
    return f"{prefix}_{mid}_{safe_stamp()}.csv"
def validate_df(df: pd.DataFrame,
                required_cols: List[str],
                dtypes_map: Dict[str, str]) -> Dict[str, str]:

    msgs = {}

    missing = [c for c in required_cols if c not in df.columns]

    if missing:
        msgs['missing_cols'] = f"Missing columns: {missing}"

    for col, dtype in dtypes_map.items():
        if col in df.columns:
            try:
                if dtype == 'datetime64[ns]':
                    pd.to_datetime(df[col])
                elif dtype == 'float':
                    pd.to_numeric(df[col])
            except Exception as e:
                msgs[f'dtype_{col}'] = (
                    f"Failed to coerce {col} to {dtype}: {e}"
                )

    na_counts = df.isna().sum().sum()
    msgs['na_total'] = f"Total NA values: {na_counts}"

    return msgs

In [6]:
SYMBOL = "MRK"

use_alpha = bool(ALPHA_KEY)

print("Using Alpha Vantage:", use_alpha)

if use_alpha:
    url = "https://www.alphavantage.co/query"

    params = {
        "function": "TIME_SERIES_DAILY",
        "symbol": SYMBOL,
        "outputsize": "compact",
        "apikey": ALPHA_KEY,
        "datatype": "json"
    }

    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()

    js = r.json()

    key = [k for k in js.keys() if "Time Series" in k]

    if not key:
        print(
            "Alpha Vantage returned no series:",
            str(list(js.values())[0])[:150]
        )
        use_alpha = False

if use_alpha:
    series = js[key[0]]

    df_api = (
        pd.DataFrame(series).T
        .rename_axis("date")
        .reset_index()
    )

    df_api = (
        df_api[['date', '4. close']]
        .rename(columns={'4. close': 'close'})
    )

    df_api['date'] = pd.to_datetime(df_api['date'])
    df_api['close'] = pd.to_numeric(df_api['close'])

if not use_alpha:
    import yfinance as yf

    df_api = (
        yf.download(
            SYMBOL,
            period="6mo",
            interval="1d",
            auto_adjust=False,
            multi_level_index=False
        )
        .reset_index()[['Date', 'Close']]
    )

    df_api.columns = ['date', 'close']

df_api = df_api.sort_values('date').reset_index(drop=True)
print(df_api.head())
print("Shape:", df_api.shape)
print("NA counts:")
print(df_api.isna().sum())

msgs = validate_df(
    df_api,
    required_cols=['date', 'close'],
    dtypes_map={
        'date': 'datetime64[ns]',
        'close': 'float'
    }
)

print(msgs)


Using Alpha Vantage: True
        date   close
0 2026-03-30  118.10
1 2026-03-31  120.29
2 2026-04-01  120.84
3 2026-04-02  120.87
4 2026-04-06  120.85
Shape: (100, 2)
NA counts:
date     0
close    0
dtype: int64
{'na_total': 'Total NA values: 0'}


In [7]:
fname = safe_filename(
    prefix="api",
    meta={
        "source": "alpha" if use_alpha else "yfinance",
        "symbol": SYMBOL
    }
)

out_path = DATA_RAW / fname

df_api.to_csv(out_path, index=False)

print("Saved:", out_path)

Saved: data\raw\api_source-alpha_symbol-MRK_20260821-054104.csv


In [11]:
SCRAPE_URL = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"

headers = {
    "User-Agent": "AFE-Course-Notebook/1.0"
}

resp = requests.get(
    SCRAPE_URL,
    headers=headers,
    timeout=30
)

resp.raise_for_status()

soup = BeautifulSoup(resp.text, "html.parser")

table = soup.find('table', id='constituents')

if table is None:
    raise RuntimeError("Could not find table with id='constituents'")

rows = []

for tr in table.find_all("tr"):
    cells = [
        td.get_text(strip=True)
        for td in tr.find_all(["td", "th"])
    ]

    if cells:
        rows.append(cells)

header = rows[0]
data = rows[1:]

df_scrape = pd.DataFrame(data, columns=header)

print(df_scrape.head())
print("Shape:", df_scrape.shape)
print("NA counts:")
print(df_scrape.isna().sum())

  Symbol             Security              GICSSector  \
0    MMM                   3M             Industrials   
1    AOS          A. O. Smith             Industrials   
2    ABT  Abbott Laboratories             Health Care   
3   ABBV               AbbVie             Health Care   
4    ACN            Accenture  Information Technology   

                GICS Sub-Industry    Headquarters Location  Date added  \
0        Industrial Conglomerates    Saint Paul, Minnesota  1957-03-04   
1               Building Products     Milwaukee, Wisconsin  2017-07-26   
2           Health Care Equipment  North Chicago, Illinois  1957-03-04   
3                   Biotechnology  North Chicago, Illinois  2012-12-31   
4  IT Consulting & Other Services          Dublin, Ireland  2011-07-06   

          CIK      Founded  
0  0000066740         1902  
1  0000091142         1916  
2  0000001800         1888  
3  0001551152  2013 (1888)  
4  0001467373         1989  
Shape: (503, 8)
NA counts:
Symbol     

In [12]:
msgs2 = validate_df(
    df_scrape,
    required_cols=list(df_scrape.columns),
    dtypes_map={}
)

print(msgs2)

{'na_total': 'Total NA values: 0'}


In [13]:
fname2 = safe_filename(
    prefix="scrape",
    meta={
        "site": "wikipedia",
        "table": "S&P500"
    }
)

out_path2 = DATA_RAW / fname2

df_scrape.to_csv(out_path2, index=False)

print("Saved:", out_path2)

Saved: data\raw\scrape_site-wikipedia_table-S&P500_20260821-061642.csv


API source: Alpha Vantage TIME_SERIES_DAILY, with yfinance as fallback.
Ticker: MRK.
Output size: compact.

Scraping source: List of S&P 500 companies Wikipedia page.
Table selected: components table.

## Validation

API data:
- Required columns: date, close
- Date converted to datetime
- Close converted to numeric
- Shape and missing values inspected

Scraped data:
- Confirmed expected table columns
- Inspected shape and missing values
- Converted applicable numeric columns

## Assumptions & Risks

- API schemas may change.
- Alpha Vantage may impose rate limits or return non-data responses.
- Website HTML structure can change and break scraping selectors.
- Scraped information depends on the source page being current.
- API credentials are stored in .env and are not committed to GitHub.